# Data Exploration Notebook

---
embed-resources: true
---

## Methods

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime

### Data

In [ ]:
# load data from excel
xlsx = pd.ExcelFile("../data/raw/datathon_data.xlsx")

portfolios = ['A', 'B', 'C', 'D']

daily_dfs = {}
interval_dfs = {}

for p in portfolios:
    d = pd.read_excel(xlsx, f'{p} - Daily')
    d.columns = [c.strip() for c in d.columns]
    d['Date'] = pd.to_datetime(d['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    d['client'] = p
    daily_dfs[p] = d

    iv = pd.read_excel(xlsx, f'{p} - Interval')
    iv.columns = [c.strip() for c in iv.columns]
    iv['client'] = p
    interval_dfs[p] = iv

daily_df = pd.concat(daily_dfs.values(), ignore_index=True)
interval_df = pd.concat(interval_dfs.values(), ignore_index=True)
staffing_df = pd.read_excel(xlsx, 'Daily Staffing')

print("daily:", daily_df.shape)
print("interval:", interval_df.shape)
print("staffing:", staffing_df.shape)
daily_df.head()

#### Data Cleaning

In [ ]:
# clean up the numeric cols
numeric_cols = ['Call Volume', 'CCT', 'Service Level', 'Abandon Rate']
for col in numeric_cols:
    if col in daily_df.columns:
        daily_df[col] = pd.to_numeric(daily_df[col], errors='coerce')
    if col in interval_df.columns:
        interval_df[col] = pd.to_numeric(interval_df[col], errors='coerce')

# get rid of negative values (doesnt make sense for call volume / CCT)
daily_df = daily_df[daily_df['Call Volume'].isna() | (daily_df['Call Volume'] >= 0)]
daily_df = daily_df[daily_df['CCT'].isna() | (daily_df['CCT'] >= 0)]

# service level and abandon rate should be between 0 and 1
daily_df['Service Level'] = daily_df['Service Level'].clip(0, 1)
daily_df['Abandon Rate'] = daily_df['Abandon Rate'].clip(0, 1)

# fill gaps
daily_df = daily_df.groupby('client', group_keys=False).apply(lambda x: x.ffill().bfill())
interval_df = interval_df.groupby('client', group_keys=False).apply(lambda x: x.ffill().bfill())

# add some time features for later
daily_df['day_of_week'] = daily_df['Date'].dt.dayofweek
daily_df['month'] = daily_df['Date'].dt.month
daily_df['day_of_month'] = daily_df['Date'].dt.day
daily_df['week_of_year'] = daily_df['Date'].dt.isocalendar().week.astype(int)
daily_df['quarter'] = daily_df['Date'].dt.quarter
daily_df['is_weekend'] = daily_df['day_of_week'].isin([5, 6]).astype(int)

# lag features - tried a bunch, these seemed useful
daily_df = daily_df.sort_values(['client', 'Date']).reset_index(drop=True)
for lag in [1, 2, 7, 14, 21, 28]:
    daily_df[f'Call Volume_lag_{lag}'] = daily_df.groupby('client')['Call Volume'].shift(lag)

# rolling stats
for w in [3, 7, 14, 30]:
    shifted = daily_df.groupby('client')['Call Volume'].shift(1)
    daily_df[f'Call Volume_rollmean_{w}'] = shifted.groupby(daily_df['client']).rolling(w).mean().reset_index(level=0, drop=True)
    daily_df[f'Call Volume_rollstd_{w}'] = shifted.groupby(daily_df['client']).rolling(w).std().reset_index(level=0, drop=True)

# some interaction features
daily_df['diff_lag7'] = daily_df['Call Volume'] - daily_df['Call Volume_lag_7']
lag7 = daily_df.groupby('client')['Call Volume'].shift(7)
daily_df['pct_change_7'] = ((daily_df['Call Volume'] - lag7) / lag7.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0).clip(-5, 5)

daily_df['volatility_ratio'] = (daily_df['Call Volume_rollstd_7'] / daily_df['Call Volume_rollmean_7'].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)

daily_df['volume_x_service'] = daily_df['Call Volume'] * daily_df['Service Level']
daily_df['volume_x_abandon'] = daily_df['Call Volume'] * daily_df['Abandon Rate']

daily_df['trend'] = daily_df.groupby('client').cumcount()

print(daily_df.shape, interval_df.shape)

### Summary Statistics

In [ ]:
# quick look at averages
print("Avg Call Volume by Month:")
print(daily_df.groupby(['client', 'month'])['Call Volume'].mean().unstack(level=0))

print("\nAvg Call Volume by Quarter:")
print(daily_df.groupby(['client', 'quarter'])['Call Volume'].mean().unstack(level=0))

print("\nStd Dev:")
print(daily_df.groupby('client')['Call Volume'].std())

print("\nVolume-CCT correlation:")
print(daily_df.groupby('client')[['Call Volume', 'CCT']].corr())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("../outputs/outlier_checks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IQR_MULTIPLIER = 1.5
EXTREME_IQR_MULTIPLIER = 3.0
Z_THRESHOLD = 4

def analyze_outliers(df, df_name):
    print(f"\n--- Analyzing: {df_name} ---")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns: {len(numeric_cols)}")

    all_flagged_rows = []

    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) == 0:
            continue

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - IQR_MULTIPLIER * IQR
        upper_bound = Q3 + IQR_MULTIPLIER * IQR
        extreme_lower = Q1 - EXTREME_IQR_MULTIPLIER * IQR
        extreme_upper = Q3 + EXTREME_IQR_MULTIPLIER * IQR

        iqr_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        extreme_outliers = df[(df[col] < extreme_lower) | (df[col] > extreme_upper)]

        # z-score
        z_scores = (series - series.mean()) / series.std()
        z_outliers = df.loc[z_scores[np.abs(z_scores) > Z_THRESHOLD].index]

        if len(extreme_outliers) > 0:
            print(f"  {col}: {len(iqr_outliers)} IQR / {len(extreme_outliers)} extreme / {len(z_outliers)} z-score outliers")
            extreme_outliers = extreme_outliers.copy()
            extreme_outliers["outlier_column"] = col
            all_flagged_rows.append(extreme_outliers)
            extreme_outliers.to_csv(OUTPUT_DIR / f"{df_name}_{col}_extreme_outliers.csv", index=False)

    if all_flagged_rows:
        combined = pd.concat(all_flagged_rows).drop_duplicates()
        combined.to_csv(OUTPUT_DIR / f"{df_name}_ALL_flagged_rows.csv", index=False)
        print(f"  Saved {len(combined)} flagged rows total")
    else:
        print("  No extreme outliers found")

analyze_outliers(daily_df, "daily_df")
analyze_outliers(interval_df, "interval_df")
analyze_outliers(staffing_df, "staffing_df")

In [110]:
df_flagged = pd.read_csv(
    "../outputs/outlier_checks/interval_df_ALL_flagged_rows.csv"
)

print(df_flagged.describe())
print(df_flagged.head(10))

                Day  Service Level   Call Volume  Abandoned Calls  \
count  10013.000000   10013.000000  10013.000000     10013.000000   
mean      15.594927       0.808984    379.883252         8.072406   
std        8.937597       0.234325    351.023955        16.622890   
min        1.000000       0.000000      0.000000         0.000000   
25%        8.000000       0.653000     38.000000         0.000000   
50%       16.000000       0.932700    313.000000         2.000000   
75%       23.000000       0.996000    591.000000        10.000000   
max       31.000000       1.000000   1200.000000       238.000000   

       Abandoned Rate           CCT          hour        minute    time_index  \
count    10013.000000  10013.000000  10013.000000  10013.000000  10013.000000   
mean         0.044463    343.460067     12.649556     15.927295     25.830021   
std          0.103118    147.960780      6.378159     14.972058     12.747849   
min          0.000000      0.000000      0.000000     

In [ ]:
# check correlations for each client
for client in daily_df['client'].unique():
    sub = daily_df[daily_df['client'] == client]
    corr = sub.corr(numeric_only=True)
    print(f"\nClient {client} correlation matrix:")
    print(corr)

In [112]:
print(interval_df["Service Level"].describe())
print(interval_df["Abandoned Rate"].describe())
print(interval_df["Call Volume"].describe())
print(interval_df["CCT"].describe())

count    17102.000000
mean         0.926251
std          0.141879
min          0.000000
25%          0.922200
50%          0.989200
75%          1.000000
max          1.000000
Name: Service Level, dtype: float64
count    17102.000000
mean         0.015720
std          0.060796
min          0.000000
25%          0.000000
50%          0.000000
75%          0.008900
max          1.000000
Name: Abandoned Rate, dtype: float64
count    17102.000000
mean       213.299380
std        249.162017
min          0.000000
25%         18.000000
50%        127.000000
75%        327.000000
max       1200.000000
Name: Call Volume, dtype: float64
count    17102.000000
mean       319.687060
std        114.951599
min          0.000000
25%        286.622500
50%        318.310000
75%        342.257500
max       4786.000000
Name: CCT, dtype: float64


In [113]:
print(interval_df.loc[
    interval_df["CCT"] > 3000,
    ["Month", "Day", "Interval", "CCT", "Call Volume"]
])

       Month  Day  Interval     CCT  Call Volume
4592    June    4  05:00:00  3270.0          1.0
15500  April    5  04:00:00  4786.0          1.0
15535  April    5  05:00:00  4786.0          0.0


In [114]:
CCT_CAP = 1500

interval_df["CCT"] = interval_df["CCT"].clip(
    upper=CCT_CAP
)

daily_df["CCT"] = daily_df["CCT"].clip(
    upper=CCT_CAP
)

interval_df["CCT"].describe()

count    17102.000000
mean       318.751190
std         95.953372
min          0.000000
25%        286.622500
50%        318.310000
75%        342.257500
max       1500.000000
Name: CCT, dtype: float64

In [ ]:
# which features correlate most with call volume?
for client in daily_df['client'].unique():
    sub = daily_df[daily_df['client'] == client]
    corr = sub.corr(numeric_only=True)['Call Volume'].abs().sort_values(ascending=False)
    print(f"\nClient {client} - top 15:")
    print(corr[1:16])

### Exploratory Visualization

In [ ]:
# top feature correlations heatmap
for client in daily_df['client'].unique():
    sub = daily_df[daily_df['client'] == client]
    top_corr = sub.corr(numeric_only=True)['Call Volume'].abs().sort_values(ascending=False).head(20)
    cols = top_corr.index
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(sub[cols].corr(), annot=False, cmap="coolwarm", center=0)
    plt.title(f"Top Feature Correlations - Client {client}")
    plt.savefig(f"../outputs/client_{client}/top_features_corr_client.png", bbox_inches="tight")
    plt.show()

In [ ]:
# avg volume by day of week
for client in portfolios:
    sub = daily_df[daily_df['client'] == client]
    avg = sub.groupby("day_of_week")["Call Volume"].mean()
    
    plt.figure()
    avg.plot(kind="bar")
    plt.title(f"Avg Volume by DOW - {client}")
    plt.xlabel("Day (0=Mon)")
    plt.ylabel("Call Volume")
    plt.savefig(f"../outputs/client_{client}/avg_volume_by_day_of_week.png")
    plt.show()

In [ ]:
# service level vs abandon rate - are they inversely related?
for client in portfolios:
    sub = daily_df[daily_df['client'] == client]
    plt.figure()
    plt.scatter(sub["Service Level"], sub["Abandon Rate"], alpha=0.5)
    plt.title(f"Service Level vs Abandon Rate - {client}")
    plt.xlabel("Service Level")
    plt.ylabel("Abandon Rate")
    plt.savefig(f"../outputs/client_{client}/service_vs_abandon.png")
    plt.show()

In [ ]:
# rolling average to see trends
for client in portfolios:
    sub = daily_df[daily_df['client'] == client].sort_values("Date")
    roll = sub["Call Volume"].rolling(7).mean()
    plt.figure()
    roll.plot()
    plt.title(f"7-Day Rolling Avg - {client}")
    plt.savefig(f"../outputs/client_{client}/rolling_7day_volume.png")
    plt.show()

In [ ]:
# does lag 7 actually predict current volume?
for client in portfolios:
    sub = daily_df[daily_df['client'] == client]
    plt.figure()
    plt.scatter(sub["Call Volume_lag_7"], sub["Call Volume"], alpha=0.4, s=10)
    plt.title(f"Lag7 vs Current - {client}")
    plt.xlabel("Last Week")
    plt.ylabel("This Week")
    plt.savefig(f"../outputs/client_{client}/lag7_relationship.png")
    plt.show()

In [ ]:
import os

daily_df["Date"] = pd.to_datetime(daily_df["Date"])

for client in daily_df["client"].unique():
    df_client = daily_df[daily_df["client"] == client].copy()

    # add day/month names just for this heatmap
    df_client["day_name"] = df_client["Date"].dt.day_name()
    df_client["month_name"] = df_client["Date"].dt.month_name()

    day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    month_order = ["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"]

    heatmap_data = (
        df_client
        .groupby(["day_name", "month_name"])["Call Volume"]
        .mean()
        .unstack()
        .reindex(index=day_order, columns=month_order)
    )

    plt.figure(figsize=(12, 6))
    sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlGnBu",
                linewidths=0.3, cbar_kws={"label": "Average Call Volume"})
    plt.title(f"Average Call Volume by Day and Month — Client {client}")
    plt.xlabel("Month")
    plt.ylabel("Day of Week")
    plt.tight_layout()

    output_dir = f"../outputs/client_{client}"
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f"{output_dir}/heatmap_call_volume_day_month.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# monthly breakdown
for client in portfolios:
    sub = daily_df[daily_df['client'] == client]
    sub.groupby("month")["Call Volume"].mean().plot(kind="bar")
    plt.title(f"Avg Volume by Month - {client}")
    plt.tight_layout()
    plt.show()

In [ ]:
# volatility over time
for client in portfolios:
    sub = daily_df[daily_df['client'] == client].sort_values("Date")
    sub["Call Volume"].rolling(7).std().plot()
    plt.title(f"Rolling Std - {client}")
    plt.savefig(f"../outputs/client_{client}/rolling_std_7.png")
    plt.show()

features to maybe drop later:
- volume_x_service
- volume_x_abandon
- lag7_x_weekend
- trend_squared
- month
- quarter

In [ ]:
print(daily_df.shape, interval_df.shape)
print("nulls:", daily_df.isna().sum().sum())

In [125]:
print(daily_df.shape)
print(interval_df.shape)
print(staffing_df.shape)

(2924, 64)
(17102, 41)
(365, 5)


In [ ]:
# save for the feature engineering notebook
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

daily_df.to_csv(out / "daily_df.csv", index=False)
interval_df.to_csv(out / "interval_df.csv", index=False)
staffing_df.to_csv(out / "staffing_df.csv", index=False)
print("done")